# ⚙️ Notebook 2 — Preprocessing & Feature Engineering
**Kelompok 1 — Tugas Besar Analisis Big Data**

Tahapan:
1. Imputasi nilai NULL dengan rata-rata kolom
2. Penghapusan duplikat
3. VectorAssembler (gabungkan fitur)
4. StandardScaler (normalisasi)
5. Split dataset 80% training / 20% testing

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean as spark_mean, count, when, isnan
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName('WaterPotability_Preprocessing') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

print('SparkSession berhasil dibuat.')

---
## 2.1 Membaca Data Mentah

In [ ]:
DATA_PATH = '/home/jovyan/work/data/water_potability.csv'

df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
print(f'Jumlah baris awal: {df.count():,}')
df.show(3)

---
## 2.2 Imputasi Missing Values (Mean Imputation)

In [ ]:
# Kolom yang memiliki nilai NULL
cols_with_null = ['ph', 'Sulfate', 'Trihalomethanes']

print('=== IMPUTASI MEAN ===')
fill_values = {}
for col_name in cols_with_null:
    mean_val = df.select(spark_mean(col(col_name))).collect()[0][0]
    fill_values[col_name] = mean_val
    print(f'  {col_name:<20} mean = {mean_val:.4f}')

df = df.fillna(fill_values)

# Verifikasi tidak ada lagi NULL
missing_after = df.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

total_missing = sum(missing_after.values())
print(f'\nTotal missing setelah imputasi: {total_missing}')
assert total_missing == 0, 'Masih ada missing values!'
print('✅ Semua missing values berhasil diimputasi.')

---
## 2.3 Penghapusan Duplikat

In [ ]:
n_before = df.count()
df = df.dropDuplicates()
n_after = df.count()

print(f'Baris sebelum deduplikasi : {n_before:,}')
print(f'Baris sesudah deduplikasi : {n_after:,}')
print(f'Duplikat dihapus          : {n_before - n_after:,}')

---
## 2.4 VectorAssembler — Menggabungkan Fitur

In [ ]:
FEATURE_COLS = [
    'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
    'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity'
]
LABEL_COL = 'Potability'

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='features_raw',
    handleInvalid='keep'
)

df_assembled = assembler.transform(df)
print('✅ VectorAssembler berhasil. Kolom "features_raw" dibuat.')
df_assembled.select('features_raw', LABEL_COL).show(3, truncate=False)

---
## 2.5 StandardScaler — Normalisasi Fitur

In [ ]:
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print('✅ StandardScaler berhasil. Kolom "features" (normalized) dibuat.')
df_scaled.select('features', LABEL_COL).show(3, truncate=True)

---
## 2.6 Split Dataset — 80% Training / 20% Testing

In [ ]:
# Pilih hanya kolom yang diperlukan
df_final = df_scaled.select('features', LABEL_COL)

# Split dataset
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)

print('=== HASIL SPLIT DATASET ===')
print(f'Training set  : {train_df.count():,} sampel ({train_df.count()/df_final.count()*100:.1f}%)')
print(f'Testing set   : {test_df.count():,} sampel ({test_df.count()/df_final.count()*100:.1f}%)')
print(f'Total         : {df_final.count():,} sampel')

# Distribusi label di training set
print('\nDistribusi label di training set:')
train_df.groupBy(LABEL_COL).count().show()

---
## 2.7 Simpan Dataset untuk Notebook Berikutnya

In [ ]:
train_df.write.mode('overwrite').parquet('/home/jovyan/work/output/train_data.parquet')
test_df.write.mode('overwrite').parquet('/home/jovyan/work/output/test_data.parquet')

print('✅ Training set disimpan : output/train_data.parquet')
print('✅ Testing set disimpan  : output/test_data.parquet')
print('\n➡️ Lanjut ke Notebook 03: Model Training')

spark.stop()